In [1]:
import os
import folium
import wget
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

## Task 1: Marking Launch Sites

In [2]:
# Loading dataset
filepath = '../data/raw/spacex_launch_geo.csv'
if os.path.exists(filepath):
    print('File already exists, reading file...')
    spacex_df = pd.read_csv(filepath)
else:
    wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv', out='../data/raw')
    spacex_df = pd.read_csv(filepath)
    print('File downloaded and read...')

spacex_df.head()

File already exists, reading file...


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [3]:
# Column selection
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
spacex_df = spacex_df.rename(columns={'Launch Site': 'Launch_Site'})
spacex_df.head()

,Launch_Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0


- Basemap

In [4]:
# Initializing with NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
basemap = folium.Map(location=nasa_coordinate, zoom_start=4)
# NASA JSC
folium.CircleMarker(
    location=nasa_coordinate,
    radius=12, #in px
    color='blue',
    fill= True,
    popup='NASA Jhonson Space Center'
).add_to(basemap)

label_html = f"""
    <div style="
        font-size: 12px;
        font-weight: 600;
        padding: 2px 7px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.4);
        pointer-events: none"> 
        NASA JSC 
    </div>
    """

folium.Marker(
    location=nasa_coordinate,
    icon=DivIcon(
        icon_size=(85,24),
        icon_anchor=(-15, 12),
        html=label_html
    )
).add_to(basemap)
# Saving Initial Map
if not os.path.exists('../maps/basemap.html'):
    basemap.save('../maps/basemap.html')

basemap

- A circle for each Launch Site in dataframe

In [5]:
for row in spacex_df.itertuples():
    coordinates = [row.Lat, row.Long]
    # Outer ring
    folium.Circle(
        location=coordinates,
        radius=1000,
        color='blue',
        weight=1.2, 
        fill=True,
        fill_color='blue',
        fill_opacity='0.05',
        interactive=False,
        popup=folium.Popup(row.Launch_Site)
    ).add_to(basemap)
    # Center focal point
    folium.CircleMarker(
        location=coordinates,
        radius=4, #in px
        color='red',
        fill=True,
        fill_opacity='0.15',
        weight=1.5
    ).add_to(basemap)
    #Label Marker
    folium.Marker(
        location=coordinates,
        icon=DivIcon(
            icon_size=(120, 24),
            icon_anchor=(-15,15),
            html=f"""
            <div style="
            font-size: 12px;
            font-weight: 500;
            padding: 2px 7px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.07);
            pointer-events: none
            ">
            {row.Launch_Site}
            </div>
            """
        )
    ).add_to(basemap)

if not os.path.exists('../maps/launchSites.html'):
    basemap.save('../maps/launchSites.html')

basemap

## Task 2: Marking successess and failures of launches for each site on the map

In [6]:
# Creating a color column for each successes and failures to aid in marking
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'red' if x==0 else 'green')
spacex_df.tail()

,Launch_Site,Lat,Long,class,marker_color
51,CCAFS SLC-40,28.563197,-80.57682,0,red
52,CCAFS SLC-40,28.563197,-80.57682,0,red
53,CCAFS SLC-40,28.563197,-80.57682,0,red
54,CCAFS SLC-40,28.563197,-80.57682,1,green
55,CCAFS SLC-40,28.563197,-80.57682,0,red


In [7]:
# Initializing MarkerCluster() object
marker_cluster = MarkerCluster()
basemap.add_child(marker_cluster)

for row in spacex_df.itertuples():
    folium.Marker(
        location=[row.Lat, row.Long],
        popup=folium.Popup(row.Launch_Site),
        icon=folium.Icon(
            color='white',
            icon_color=row.marker_color,
            icon='info-sign'
        )
    ).add_to(marker_cluster)

if not os.path.exists('../maps/launchresults.html'):
    basemap.save('../maps/launchresults.html')

basemap

## Task 3: Calculate the distance between a launch site to its proximities

In [8]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
).add_to(basemap)

In [9]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [10]:
launch_site = [28.56321, -80.57683]

# points of interest near CCAFS_SLC-40 launch site
ccafs_slc_40_poi = {
    'coastline' : (28.5633, -80.56789),
    'railway' : (28.56345, -80.58691),
    'highway' : (28.56338, -80.58696),
    'city' : (28.0744, -80.62592),
}


def add_distance_marker(launch_site, coordinates, label):
    """Place a marker with the distance to the launch site and draw a line to it."""
    distance = calculate_distance(coordinates[0], coordinates[1], launch_site[0], launch_site[1])

    folium.Marker(
        location=[coordinates[0], coordinates[1]],
        tooltip=label,
        icon=DivIcon(
            icon_size=(120,20),
            icon_anchor=(0,0),
            html=f'''
                <div style="
                    font-size: 12px; 
                    color:#d35400;">
                    <b>{distance:10.2f}KM</b>
                </div>
            '''
        )
    ).add_to(basemap)  

    folium.PolyLine(
        locations=[(coordinates[0], coordinates[1]),(launch_site[0], launch_site[1])],
        weight=2
    ).add_to(basemap)


for label, coordinates in ccafs_slc_40_poi.items():
    add_distance_marker(launch_site, coordinates, label)

if not os.path.exists('..maps/distance_poi.html'):
    basemap.save('../maps/distance_poi.html')

basemap